In [ ]:
# Download Packages
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad
import annoy

from plotnine import *
import matplotlib.pyplot as plt
import seaborn as sb
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

sc.set_figure_params(figsize=(5,5)) # no blurry figures allowed

In [ ]:
# Load ATAC high-quality-rna-filtered samples
atac_dir = '/Genomics/pritykinlab/seth/Diet_WL_scMultiome/Diet_GLP_scMultiome_Scripts/atac_filtered/'

nk_sfd_atac = sc.read_h5ad(atac_dir + 'NK_SFD_atac.h5ad')
nk_hfd_atac = sc.read_h5ad(atac_dir + 'NK_HFD_atac.h5ad')
nk_glp_atac = sc.read_h5ad(atac_dir + 'NK_GLP_atac.h5ad')
nk_cr_atac  = sc.read_h5ad(atac_dir + 'NK_CR_atac.h5ad')


In [ ]:
# Compare Cell Ranger FRiP vs Atlas FRiP for GEX-filtered cells across all conditions
cr_base     = '/Genomics/pritykinlab/seth/Diet_WL_scMultiome/cr_arc_outputs/'
barcode_dir = '/Genomics/pritykinlab/seth/Diet_WL_scMultiome/Diet_GLP_scMultiome_Scripts/gex_filtered/barcodes/'

cr_samples = {
    'NK_SFD': 'NK_1-SFD',
    'NK_HFD': 'NK_2-HFD',
    'NK_GLP': 'NK_3-HFD_GLP',
    'NK_CR':  'NK_4-HFD_CR',
}

atac_objects = {
    'NK_SFD': nk_sfd_atac,
    'NK_HFD': nk_hfd_atac,
    'NK_GLP': nk_glp_atac,
    'NK_CR':  nk_cr_atac,
}

records = []
for short_name, cr_name in cr_samples.items():
    per_barcode = pd.read_csv(f'{cr_base}{cr_name}/outs/per_barcode_metrics.csv')
    barcodes    = pd.read_csv(f'{barcode_dir}{short_name}_barcodes.txt',
                              header=None, names=['barcode'])
    pb          = per_barcode[per_barcode['barcode'].isin(barcodes['barcode'])]

    cr_frip    = pb['atac_peak_region_fragments'].sum() / max(pb['atac_fragments'].sum(), 1)
    atlas_frip = atac_objects[short_name].obs['FRiP'].median()

    records.append({
        'Sample':     short_name,
        'N Cells':    len(pb),
        'CR FRiP':    round(cr_frip,    3),
        'Atlas FRiP': round(atlas_frip, 3),
        'Delta':      round(atlas_frip - cr_frip, 3),
    })

comparison_df = pd.DataFrame(records)
print(comparison_df.to_string(index=False))

# Bar plot
samples = comparison_df['Sample'].tolist()
x       = np.arange(len(samples))
width   = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars_cr    = ax.bar(x - width/2, comparison_df['CR FRiP'],    width, label='Cell Ranger FRiP', color='steelblue', edgecolor='black', linewidth=0.5)
bars_atlas = ax.bar(x + width/2, comparison_df['Atlas FRiP'], width, label='Atlas FRiP',       color='salmon',    edgecolor='black', linewidth=0.5)

for bar in bars_cr:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars_atlas:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(samples)
ax.set_ylabel('FRiP Score')
ax.set_ylim(0, 1.0)
ax.set_title('Cell Ranger vs Atlas FRiP\n(GEX-filtered cells only)', fontsize=13, fontweight='bold')
ax.legend()
ax.axhline(0.15, c='grey', linestyle='--', linewidth=0.8, label='min threshold (0.15)')

plt.tight_layout()
plt.savefig('frip_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Investigating Structure of ATAC Data
for name, adata in [('NK_SFD', nk_sfd_atac), ('NK_HFD', nk_hfd_atac), 
                    ('NK_GLP', nk_glp_atac), ('NK_CR',  nk_cr_atac)]:
    print(f"\n--- {name} ---")
    print(adata)
    print(adata.var.iloc[:3, :5].to_string())
    print("\nPeak matrix (5 cells x 5 peaks):")
    print(pd.DataFrame(
        adata.X[:5, :5].toarray(),
        index=adata.obs_names[:5],
        columns=adata.var_names[:5]
    ).to_string())
    print("\nQC metrics (5 cells):")
    print(adata.obs.iloc[:5].to_string())

In [ ]:
# Bar plot with # of cells and peaks per ATAC sample
samples = ['NK_SFD', 'NK_HFD', 'NK_GLP', 'NK_CR']
atac_objects = [nk_sfd_atac, nk_hfd_atac, nk_glp_atac, nk_cr_atac]

cells = [adata.n_obs  for adata in atac_objects]
peaks = [adata.n_vars for adata in atac_objects]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

metrics = [
    ('Cells', cells, 'steelblue'),
    ('Peaks', peaks, 'salmon'),
]

for ax, (title, values, color) in zip(axes, metrics):
    ax.bar(samples, values, color=color, edgecolor='black', linewidth=0.5)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel('Count')
    ax.set_xticklabels(samples, rotation=45, ha='right')
    ax.set_ylim(0, max(values) * 1.15)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, _: f'{int(val):,}'))
    for i, v in enumerate(values):
        ax.text(i, v + max(values)*0.01, f'{v:,}', ha='center', fontsize=9)

plt.suptitle('NK Cell ATAC Data Summary by Sample', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('atac_sample_summary_barplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Helper ATAC QC Functions:

def atac_qc(adata, sample_name):
    # violin plots of all ATAC QC metrics + summary stats to inform threshold selection
    metrics = ['total_insertions', 'FRiP', 'tsse', 'nucleosomal_signal', 'n_features_per_cell']
    labels  = ['Total Insertions', 'FRiP', 'TSS Enrichment', 'Nucleosomal Signal', 'Features Per Cell']

    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    fig.suptitle(f'{sample_name} ATAC QC Metrics', fontsize=14, fontweight='bold')
    for ax, metric, label in zip(axes, metrics, labels):
        ax.violinplot(adata.obs[metric], showmedians=True)
        ax.set_title(label, fontsize=10)
        ax.set_xticks([])
        ax.set_ylabel('Value')
    plt.tight_layout()
    plt.show()

    def print_stats(values, label):
        print(f"{label}: max={values.max():.3f} | 75%={np.percentile(values, 75):.3f} | "
              f"median={np.median(values):.3f} | 25%={np.percentile(values, 25):.3f} | "
              f"min={values.min():.3f} | mean={values.mean():.3f}")

    for metric, label in zip(metrics, labels):
        print_stats(adata.obs[metric].values, label)


def atac_qc2(adata, sample_name, min_insertions, max_insertions, min_frip, min_tsse, max_nucleosomal):
    # flags cells failing any threshold and plots distributions with threshold lines
    # all metrics are independent so all thresholds are set and flagged simultaneously
    adata.obs["low_insertions"]   = adata.obs["total_insertions"]   < min_insertions
    adata.obs["high_insertions"]  = adata.obs["total_insertions"]   > max_insertions
    adata.obs["low_frip"]         = adata.obs["FRiP"]               < min_frip
    adata.obs["low_tsse"]         = adata.obs["tsse"]               < min_tsse
    adata.obs["high_nucleosomal"] = adata.obs["nucleosomal_signal"] > max_nucleosomal

    fig, axes = plt.subplots(1, 5, figsize=(25, 4))
    fig.suptitle(f'{sample_name} ATAC QC Thresholds', fontsize=14, fontweight='bold')

    # total insertions histogram
    axes[0].hist(adata.obs['total_insertions'], bins=100, color='steelblue', edgecolor='black', linewidth=0.3)
    axes[0].axvline(min_insertions, c='r', linestyle='--', label=f'min={min_insertions:,}')
    axes[0].axvline(max_insertions, c='b', linestyle='--', label=f'max={max_insertions:,}')
    axes[0].set_xlabel('Total Insertions')
    axes[0].set_ylabel('Cell Count')
    axes[0].set_title('Total Insertions')
    axes[0].legend(fontsize=7)

    # FRiP histogram
    axes[1].hist(adata.obs['FRiP'], bins=100, color='salmon', edgecolor='black', linewidth=0.3)
    axes[1].axvline(min_frip, c='r', linestyle='--', label=f'min={min_frip}')
    axes[1].set_xlabel('FRiP')
    axes[1].set_ylabel('Cell Count')
    axes[1].set_title('FRiP')
    axes[1].legend(fontsize=7)

    # TSS enrichment histogram
    axes[2].hist(adata.obs['tsse'], bins=100, color='mediumpurple', edgecolor='black', linewidth=0.3)
    axes[2].axvline(min_tsse, c='r', linestyle='--', label=f'min={min_tsse}')
    axes[2].set_xlabel('TSS Enrichment Score')
    axes[2].set_ylabel('Cell Count')
    axes[2].set_title('TSS Enrichment')
    axes[2].legend(fontsize=7)

    # nucleosomal signal histogram
    axes[3].hist(adata.obs['nucleosomal_signal'], bins=100, color='seagreen', edgecolor='black', linewidth=0.3)
    axes[3].axvline(max_nucleosomal, c='r', linestyle='--', label=f'max={max_nucleosomal}')
    axes[3].set_xlabel('Nucleosomal Signal')
    axes[3].set_ylabel('Cell Count')
    axes[3].set_title('Nucleosomal Signal')
    axes[3].legend(fontsize=7)

    # scatter: total insertions vs features per cell colored by TSS score
    sc_plot = axes[4].scatter(
        adata.obs['total_insertions'],
        adata.obs['n_features_per_cell'],
        c=adata.obs['tsse'], cmap='viridis', s=1, alpha=0.5
    )
    axes[4].axvline(min_insertions, c='r', linestyle='--')
    axes[4].axvline(max_insertions, c='b', linestyle='--')
    axes[4].set_xlabel('Total Insertions')
    axes[4].set_ylabel('Features Per Cell')
    axes[4].set_title('Insertions vs Features\n(colored by TSS)')
    plt.colorbar(sc_plot, ax=axes[4], label='TSS Score')

    plt.tight_layout()
    plt.show()

    # print number of cells flagged per filter
    print(f"  low_insertions:   {adata.obs['low_insertions'].sum():,} cells flagged")
    print(f"  high_insertions:  {adata.obs['high_insertions'].sum():,} cells flagged")
    print(f"  low_frip:         {adata.obs['low_frip'].sum():,} cells flagged")
    print(f"  low_tsse:         {adata.obs['low_tsse'].sum():,} cells flagged")
    print(f"  high_nucleosomal: {adata.obs['high_nucleosomal'].sum():,} cells flagged")


def atac_filter_cells(adata):
    # removes all cells flagged by atac_qc2 in a single pass
    print("Prior")
    print(adata.shape)
    adata = adata[~adata.obs["low_insertions"]]
    print("After low insertion filter:")
    print(adata.shape)
    adata = adata[~adata.obs["high_insertions"]]
    print("After high insertion filter:")
    print(adata.shape)
    adata = adata[~adata.obs["low_frip"]]
    print("After low FRiP filter:")
    print(adata.shape)
    adata = adata[~adata.obs["low_tsse"]]
    print("After low TSS filter:")
    print(adata.shape)
    adata = adata[~adata.obs["high_nucleosomal"]]
    print("After high nucleosomal signal filter:")
    print(adata.shape)
    return adata


In [ ]:
# Violins

In [ ]:
# Histograms

In [ ]:
# Scatters